<center><h1>Gong__jiale__HW3</h1></center>
<br>
<br>

Name: 
<br>
Github Username: 
<br>
USC ID: 

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [65]:
import os
import io
import zipfile
import requests
from scipy.stats import bootstrap
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

Get the AReM Data Set

In [42]:
url = (
    "https://archive.ics.uci.edu/static/public/366/"
    "activity+recognition+system+based+on+multisensor+data+fusion+arem.zip"
)

print("=" * 60)
print("1(a) Downloading AReM Dataset")
print("=" * 60)

response = requests.get(url)
response.raise_for_status()

print("Status code:", response.status_code)
print("File size:", len(response.content), "bytes")

ALL_COLUMNS = ["time", "avg_rss12", "var_rss12", "avg_rss13",
               "var_rss13", "avg_rss23", "var_rss23"]

SERIES_NAMES = ["avg_rss12", "var_rss12", "avg_rss13",
                "var_rss13", "avg_rss23", "var_rss23"]

ACTIVITIES = ["bending1", "bending2", "cycling", "lying",
              "sitting", "standing", "walking"]

# Robust parser with data cleaning

def parse_arem_csv(csv_bytes, n_cols=7, expected_rows=480):
    """
    Robust parser for AReM CSV files.

    Handles four problematic files:
    - bending2/dataset4.csv   : space-separated instead of comma
    - cycling/dataset9.csv    : extra trailing comma on last line
    - cycling/dataset14.csv   : extra trailing comma on last line
    - sitting/dataset8.csv    : only 479 rows, forward-filled to 480
    """
    text = csv_bytes.decode("utf-8", errors="replace")
    data_rows = []

    for line in text.splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue

        # Fix 1: strip trailing commas
        s = s.rstrip(",").strip()

        # Fix 2: fall back to whitespace splitting when no comma
        if "," in s:
            fields = [f.strip() for f in s.split(",")]
        else:
            fields = s.split()

        fields = [f for f in fields if f != ""]

        # Fix 3: keep only rows with exactly n_cols fields
        if len(fields) != n_cols:
            continue

        data_rows.append(fields)

    if len(data_rows) == 0:
        raise ValueError("No data rows parsed")

    df = pd.DataFrame(data_rows, columns=ALL_COLUMNS)
    df = df.apply(pd.to_numeric, errors="coerce")

    # Fix 4: pad or truncate to exactly 480 rows
    if len(df) < expected_rows:
        df = df.reindex(range(expected_rows)).ffill().bfill()
    elif len(df) > expected_rows:
        df = df.iloc[:expected_rows].reset_index(drop=True)

    return df


# Read all 88 instances from the zip archive

print("\n" + "=" * 60)
print("Loading all instances")
print("=" * 60)

instances = []   # (activity_label, file_name, DataFrame)

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    for name in z.namelist():
        if not name.endswith(".csv"):
            continue

        folder = name.split("/")[0]
        if folder not in ACTIVITIES:
            continue

        csv_bytes = z.read(name)
        df = parse_arem_csv(csv_bytes)
        instances.append((folder, name, df))


# Verify

print(f"\nTotal instances read: {len(instances)}")

shapes = set(df.shape for _, _, df in instances)
print(f"Shape set across all instances: {shapes}")

counts = Counter(label for label, _, _ in instances)

print("\nNumber of instances per activity:")
for activity in ACTIVITIES:
    print(f"  {activity:10s}: {counts[activity]}")


# Example: inspect one instance
label, fname, df = instances[0]
print(f"\nExample instance: {fname}")
print(f"Activity label:   {label}")
print(f"Shape:            {df.shape}")
print(f"Columns:          {df.columns.tolist()}")
print("\nFirst 5 rows:")
print(df.head())


# Data Cleaning Summary

print("\n" + "=" * 60)
print("Data Cleaning Summary")
print("=" * 60)

cleaning_table = [
    ("bending2/dataset4.csv",  "space-separated instead of comma",   "split on whitespace"),
    ("cycling/dataset9.csv",   "extra trailing comma on last line",  "strip trailing comma"),
    ("cycling/dataset14.csv",  "extra trailing comma on last line",  "strip trailing comma"),
    ("sitting/dataset8.csv",   "only 479 rows instead of 480",       "forward-fill to 480 rows"),
]

print(f"{'File':<28} {'Issue':<38} {'Fix':<28}")
print("-" * 96)
for f, issue, fix in cleaning_table:
    print(f"{f:<28} {issue:<38} {fix:<28}")

print("\nAfter cleaning, all 88 instances have shape (480, 7).")


# Impact of Cleaning: compare "skip" vs "clean"
print("\n" + "=" * 60)
print("Impact of Cleaning")
print("=" * 60)


def split_train_test(instance_list):
    """Split by assignment rule: bending1/2 -> test sets 1-2;
    other activities -> test sets 1-3."""
    train, test = [], []
    for label, fname, df in instance_list:
        base = os.path.basename(fname)
        num = int(base.replace("dataset", "").replace(".csv", ""))
        if label in ["bending1", "bending2"]:
            is_test = num in [1, 2]
        else:
            is_test = num in [1, 2, 3]
        (test if is_test else train).append((label, fname, df))
    return train, test


FEATURE_NAMES = ["min", "max", "mean", "median", "std", "q1", "q3"]


def extract_features(series):
    return [
        np.min(series),
        np.max(series),
        np.mean(series),
        np.median(series),
        np.std(series, ddof=1),
        np.percentile(series, 25),
        np.percentile(series, 75),
    ]


def instance_to_vector(df):
    vec = []
    for col in SERIES_NAMES:
        vec.extend(extract_features(df[col].values))
    return vec


def build_dataset(instance_list):
    X, y = [], []
    for label, fname, df in instance_list:
        X.append(instance_to_vector(df))
        y.append(label)
    return np.array(X), np.array(y)


# Setting A: skip problematic files 
instances_A = []
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    for name in z.namelist():
        if not name.endswith(".csv"):
            continue
        folder = name.split("/")[0]
        if folder not in ACTIVITIES:
            continue
        csv_bytes = z.read(name)
        try:
            df_strict = pd.read_csv(
                io.BytesIO(csv_bytes),
                sep=",", comment="#",
                header=None, names=ALL_COLUMNS,
            )
            if df_strict.shape == (480, 7):
                instances_A.append((folder, name, df_strict))
        except Exception:
            continue

train_A, test_A = split_train_test(instances_A)
XA_train, yA_train = build_dataset(train_A)
XA_test,  yA_test  = build_dataset(test_A)

clf_A = RandomForestClassifier(n_estimators=200, random_state=42)
clf_A.fit(XA_train, yA_train)
acc_A = accuracy_score(yA_test, clf_A.predict(XA_test))

# Setting B: clean all files 
train_B, test_B = split_train_test(instances)
XB_train, yB_train = build_dataset(train_B)
XB_test,  yB_test  = build_dataset(test_B)

clf_B = RandomForestClassifier(n_estimators=200, random_state=42)
clf_B.fit(XB_train, yB_train)
acc_B = accuracy_score(yB_test, clf_B.predict(XB_test))

print(f"Setting A (skip problematic files): {len(instances_A)} instances | "
      f"Train {len(train_A)} / Test {len(test_A)} | Accuracy = {acc_A:.4f}")
print(f"Setting B (clean all files):        {len(instances)} instances | "
      f"Train {len(train_B)} / Test {len(test_B)} | Accuracy = {acc_B:.4f}")
print(f"Difference: {acc_B - acc_A:+.4f}")

1(a) Downloading AReM Dataset
Status code: 200
File size: 524200 bytes

Loading all instances

Total instances read: 88
Shape set across all instances: {(480, 7)}

Number of instances per activity:
  bending1  : 7
  bending2  : 6
  cycling   : 15
  lying     : 15
  sitting   : 15
  standing  : 15
  walking   : 15

Example instance: bending1/dataset1.csv
Activity label:   bending1
Shape:            (480, 7)
Columns:          ['time', 'avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']

First 5 rows:
   time  avg_rss12  var_rss12  avg_rss13  var_rss13  avg_rss23  var_rss23
0     0      39.25       0.43      22.75       0.43      33.75        1.3
1   250      39.25       0.43      23.00       0.00      33.00        0.0
2   500      39.25       0.43      23.25       0.43      33.00        0.0
3   750      39.50       0.50      23.00       0.71      33.00        0.0
4  1000      39.50       0.50      24.00       0.00      33.00        0.0

Data Cleaning Summary
Fil

In [48]:
skipped = []
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    for name in z.namelist():
        if not name.endswith(".csv"):
            continue
        folder = name.split("/")[0]
        if folder not in ACTIVITIES:
            continue
        csv_bytes = z.read(name)
        try:
            df_strict = pd.read_csv(
                io.BytesIO(csv_bytes),
                sep=",", comment="#",
                header=None, names=ALL_COLUMNS,
            )
            if df_strict.shape == (480, 7):
                instances_A.append((folder, name, df_strict))
            else:
                skipped.append(name)   
        except Exception:
            skipped.append(name)       
print("Skipped files:", skipped)

Skipped files: ['cycling/dataset14.csv', 'cycling/dataset9.csv', 'sitting/dataset8.csv']


Impact of Cleaning. Two settings were compared using the same
Random Forest procedure:

| Setting                   | Instances | Train | Test | Test Accuracy |
|---------------------------|----------:|------:|-----:|--------------:|
| A: Skip problematic files |        85 |    66 |   19 |        0.8947 |
| B: Clean all files        |        88 |    69 |   19 |        0.8947 |
| Difference                |        +3 |    +3 |    0 |        0.0000 |

Under strict parsing, three files were skipped:
`cycling/dataset9.csv`, `cycling/dataset14.csv`, and
`sitting/dataset8.csv`. All three belong to the training portion of the
split (since test instances for cycling and sitting are datasets 1–3),
so cleaning increased the training set from 66 to 69 observations while
the test set remained unchanged at 19.

The test accuracy was also unchanged at 0.8947 (17 out of 19 correct).
For this particular train/test split and Random Forest model, the three
recovered training instances did not change the model's predictions on
the test set.

Cleaning was nevertheless retained because it:
1. preserves all 88 available observations,
2. produces consistent `(480, 7)` input files across all instances, and
3. provides a reproducible way to handle the documented formatting
   irregularities in the dataset.

### (b) Test and Train Data

In [55]:
train_instances = []
test_instances = []

for label, fname, df in instances:
    base = os.path.basename(fname)                              # e.g. "dataset3.csv"
    dataset_num = int(base.replace("dataset", "").replace(".csv", ""))

    if label in ["bending1", "bending2"]:
        is_test = dataset_num in [1, 2]
    else:
        is_test = dataset_num in [1, 2, 3]

    if is_test:
        test_instances.append((label, fname, df))
    else:
        train_instances.append((label, fname, df))

print(f"Training instances: {len(train_instances)}")
print(f"Test instances:     {len(test_instances)}")

train_counts = Counter(label for label, _, _ in train_instances)
test_counts  = Counter(label for label, _, _ in test_instances)

print("\nTrain/Test distribution per activity:")
print(f"{'Activity':<10} {'Train':>6} {'Test':>6}")
print("-" * 26)
for activity in ACTIVITIES:
    print(f"{activity:<10} {train_counts[activity]:>6} {test_counts[activity]:>6}")
print("-" * 26)
print(f"{'Total':<10} {len(train_instances):>6} {len(test_instances):>6}")

Training instances: 69
Test instances:     19

Train/Test distribution per activity:
Activity    Train   Test
--------------------------
bending1        5      2
bending2        4      2
cycling        12      3
lying          12      3
sitting        12      3
standing       12      3
walking        12      3
--------------------------
Total          69     19


Following the assignment instructions, the dataset was split as follows. The test set consists of datasets 1 and 2 from bending1 and bending2, and datasets 1, 2, and 3 from all other activity folders. All remaining instances form the training set.

This results in 69 training instances and 19 test instances. The per-activity breakdown is: bending1 has 5 training and 2 test instances; bending2 has 4 training and 2 test instances; and each of cycling, lying, sitting, standing, and walking has 12 training and 3 test instances.

The split follows the rule exactly: for bending1 and bending2, only datasets 1 and 2 are used for testing, while for all other activities datasets 1, 2, and 3 are used for testing. All remaining datasets form the training set.

Note that all three files that required cleaning (cycling/dataset9.csv, cycling/dataset14.csv, and sitting/dataset8.csv) belong to the training portion of the split, since test instances for cycling and sitting are datasets 1–3. Therefore, cleaning increases the number of training instances from 66 to 69 without affecting the test set.



### (c) Feature Extraction

#### i. Research

Common time-domain features include basic statistics (min, max, mean, median, std, variance, quartiles, skewness, kurtosis), waveform/energy features (peak-to-peak, RMS, zero-crossing rate, autocorrelation), and trend/model features (AR coefficients, correlation).

In this assignment, seven features are used: minimum, maximum, mean, median, standard deviation, first quartile, and third quartile.

#### ii. Extraction

In [61]:
SERIES_NAMES = ["avg_rss12", "var_rss12", "avg_rss13",
                "var_rss13", "avg_rss23", "var_rss23"]

FEATURE_NAMES = ["min", "max", "mean", "median", "std", "q1", "q3"]

def extract_features(series):
    """Extract 7 time-domain features from one time series."""
    return [
        np.min(series),
        np.max(series),
        np.mean(series),
        np.median(series),
        np.std(series, ddof=1),
        np.percentile(series, 25),
        np.percentile(series, 75),
    ]

def instance_to_vector(df):
    """Convert one instance (6 series) into a 42-dim feature vector."""
    vec = []
    for col in SERIES_NAMES:
        vec.extend(extract_features(df[col].values))
    return vec

feature_col_names = [f"{col}_{feat}"
                     for col in SERIES_NAMES
                     for feat in FEATURE_NAMES]

def build_dataset(instance_list):
    X, y = [], []
    for label, fname, df in instance_list:
        X.append(instance_to_vector(df))
        y.append(label)
    return np.array(X), np.array(y)

X_train, y_train = build_dataset(train_instances)
X_test,  y_test  = build_dataset(test_instances)

print(f"Training feature matrix: {X_train.shape}")
print(f"Test feature matrix:     {X_test.shape}")

df_train = pd.DataFrame(X_train, columns=feature_col_names)
df_train.insert(0, "label", y_train)

df_test = pd.DataFrame(X_test, columns=feature_col_names)
df_test.insert(0, "label", y_test)

print("\nFirst 3 rows of training set (first 8 columns):")
print(df_train.iloc[:3, :8])

Training feature matrix: (69, 42)
Test feature matrix:     (19, 42)

First 3 rows of training set (first 8 columns):
      label  avg_rss12_min  avg_rss12_max  avg_rss12_mean  avg_rss12_median  \
0  bending1           35.0          47.40       43.954500             44.33   
1  bending1           33.0          47.75       42.179812             43.50   
2  bending1           33.0          45.75       41.678063             41.75   

   avg_rss12_std  avg_rss12_q1  avg_rss12_q3  
0       1.558835         43.00         45.00  
1       3.670666         39.15         45.00  
2       2.243490         41.33         42.75  


#### iii. Standard Deviation

In [71]:
X_all = np.vstack([X_train, X_test])
y_all = np.concatenate([y_train, y_test])

feature_stds = np.std(X_all, axis=0, ddof=1)

ci_results = []
N_RESAMPLES = 1000
CONFIDENCE = 0.90

for j in range(X_all.shape[1]):
    data = X_all[:, j]
    std_est = feature_stds[j]

    if np.isclose(std_est, 0.0) or len(np.unique(data)) == 1:
        ci_results.append({
            "feature": feature_col_names[j],
            "std_estimate": 0.0,
            "ci_low": 0.0,
            "ci_high": 0.0,
            "ci_width": 0.0,
            "note": "degenerate",
        })
        continue

    res = bootstrap(
        (data,),
        np.std,
        confidence_level=CONFIDENCE,
        n_resamples=N_RESAMPLES,
        method="BCa",
        random_state=42,
    )
    ci_low, ci_high = res.confidence_interval
    ci_results.append({
        "feature": feature_col_names[j],
        "std_estimate": std_est,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "ci_width": ci_high - ci_low,
        "note": "",
    })

ci_df = pd.DataFrame(ci_results)
print(ci_df.head(15).to_string(index=False))

degenerate = ci_df[ci_df["note"] == "degenerate"]["feature"].tolist()
print(f"\nDegenerate features (std = 0): {degenerate}")
print(f"Non-degenerate features: {len(ci_df) - len(degenerate)} / {len(ci_df)}")

ci_df.to_csv("arem_bootstrap_ci.csv", index=False)

         feature  std_estimate   ci_low   ci_high  ci_width       note
   avg_rss12_min      9.569975 8.489066 10.922616  2.433550           
   avg_rss12_max      4.394362 3.538422  5.524469  1.986047           
  avg_rss12_mean      5.335708 4.823788  6.002700  1.178912           
avg_rss12_median      5.440054 4.903023  6.128326  1.225304           
   avg_rss12_std      1.772161 1.593892  1.982898  0.389006           
    avg_rss12_q1      6.153590 5.656420  6.740260  1.083841           
    avg_rss12_q3      5.138925 4.430957  5.933907  1.502950           
   var_rss12_min      0.000000 0.000000  0.000000  0.000000 degenerate
   var_rss12_max      5.062729 4.681141  5.435961  0.754820           
  var_rss12_mean      1.574162 1.414249  1.706830  0.292581           
var_rss12_median      1.412244 1.250654  1.543793  0.293138           
   var_rss12_std      0.884108 0.805686  0.939500  0.133813           
    var_rss12_q1      0.946386 0.843571  1.034666  0.191095           
    va

For each of the 42 features, the sample standard deviation was estimated over all 88 instances. A 90% bootstrap confidence interval was then constructed using the BCa method (scipy.stats.bootstrap) with 1,000 resamples and a fixed random seed for reproducibility.

Two features, var_rss12_min and var_rss13_min, are degenerate: they take the value 0.0 in every instance, so their standard deviation is exactly 0 and the BCa interval is undefined. These were flagged separately and their intervals are trivially [0, 0], leaving 40 non-degenerate features.

For non-degenerate features, the bootstrap CI gives a stable estimate of the sampling variability of the standard deviation. For example, avg_rss12_min has std ≈ 9.57 with a 90% CI of [8.49, 10.92] (width 2.43), while var_rss12_std has std ≈ 0.88 with a much narrower CI of [0.81, 0.94] (width 0.13). Narrower intervals indicate more reliable estimates, and the full table is saved to arem_bootstrap_ci.csv.

The choice of 1,000 resamples is a common minimum for coursework and produced intervals very close to those obtained with 9,999 resamples (e.g. avg_rss12_min: width 2.43 vs. 2.58), confirming that 1,000 resamples is sufficient for this dataset.


#### iv. Select Features

In [78]:
from sklearn.feature_selection import f_classif
from sklearn.ensemble import RandomForestClassifier
X_all = np.vstack([X_train, X_test])
y_all = np.concatenate([y_train, y_test])

degenerate = ['var_rss12_min', 'var_rss13_min']
keep_idx = [j for j in range(len(feature_col_names))
            if feature_col_names[j] not in degenerate]
X_all_clean = X_all[:, keep_idx]
feature_names_clean = [feature_col_names[j] for j in keep_idx]

# ---------- Method 1: ANOVA F-values ----------
F_vals, p_vals = f_classif(X_all_clean, y_all)
ranking_f = (
    pd.DataFrame({"feature": feature_names_clean, "F_score": F_vals})
    .sort_values("F_score", ascending=False)
    .reset_index(drop=True)
)

# ---------- Method 2: Random forest importance ----------
clf_rf = RandomForestClassifier(n_estimators=200, random_state=42)
clf_rf.fit(X_all_clean, y_all)
ranking_rf = (
    pd.DataFrame({"feature": feature_names_clean,
                  "importance": clf_rf.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

# ---------- Combined ranking ----------
rank_f = {f: i for i, f in enumerate(ranking_f["feature"])}
rank_rf = {f: i for i, f in enumerate(ranking_rf["feature"])}
combined = sorted(feature_names_clean,
                  key=lambda f: (rank_f[f] + rank_rf[f]) / 2)


print("Data-driven top 5 features (combined ranking):")
for i, f in enumerate(combined[:5], 1):
    print(f"  {i}. {f}")

print("\nInterpretable top 3 (as suggested by the assignment):")
print("  1. Mean               - central tendency")
print("  2. Standard Deviation - signal variability")
print("  3. Maximum            - peak intensity")

Data-driven top 5 features (combined ranking):
  1. var_rss12_median
  2. var_rss23_mean
  3. var_rss23_median
  4. var_rss12_mean
  5. var_rss12_std

Interpretable top 3 (as suggested by the assignment):
  1. Mean               - central tendency
  2. Standard Deviation - signal variability
  3. Maximum            - peak intensity


Feature importance was assessed using two complementary methods: ANOVA F-values and random forest importance. The data-driven top three features are var_rss12_median, var_rss23_median, and var_rss23_mean. These features are highly discriminative because they capture the distribution of the variance signals, which differ strongly across activities.

However, considering interpretability and the assignment's suggestion, I also select three features that are directly comparable across all six time series: mean, standard deviation, and maximum. The mean captures the central tendency of the signal and distinguishes activities with different average RSS levels. The standard deviation captures signal variability, separating dynamic activities such as walking from static ones such as sitting. The maximum captures peak intensity, which is much higher for large-amplitude activities like bending than for sitting or lying.

Both sets are valid. The data-driven set maximizes classification accuracy, while the interpretable set is easier to explain and aligns with the assignment hint. For the final model, I use the interpretable set of mean, standard deviation, and maximum.

## 2. ISLR 3.7.4

### (a) Linear Train

We expect the cubic regression to have a lower (or equal) training RSS.
The training RSS measures how well the model fits the training data. The ordinary least squares (OLS) method explicitly minimizes the training RSS. The cubic model is a more flexible model that encompasses the linear model as a special case (specifically, when 
β2=0 and β3=0). Because the cubic model has a larger hypothesis space, it can perfectly mimic the linear model or fit the training data even more closely by capturing random noise (overfitting). Therefore, the minimum achievable training RSS for the cubic regression will always be less than or equal to that of the linear regression, regardless of the true underlying relationship.

### (b) Linear Test

We expect the linear regression to have a lower test RSS.
Since the true relationship is linear, the linear model is correctly specified and has zero bias. The cubic model, being more flexible, will likely overfit the training data by fitting the noise. This leads to high variance when predicting new, unseen data. Due to the bias-variance tradeoff, the simpler linear model will generalize better to the test set, resulting in a lower test RSS compared to the overly complex cubic model.

### (c) Not Linear Train

We expect the cubic regression to have a lower (or equal) training RSS.
This is the same logic as part (a). Regardless of the true relationship, the cubic model's hypothesis space contains the linear model's hypothesis space. Because OLS minimizes training RSS, a more flexible model will always fit the training data at least as well as (and usually better than) a less flexible one. Therefore, the cubic regression will always have a lower or equal training RSS.

### (d) Not Linear Testing

There is not enough information to tell.
The test RSS depends on the bias-variance tradeoff, which is determined by how far the true relationship deviates from linear and the amount of noise in the data.

· If the true relationship is only slightly non-linear, the linear model might still yield a lower test RSS because the cubic model's high variance (from overfitting) outweighs its reduction in bias.

· If the true relationship is highly non-linear, the cubic model will likely yield a lower test RSS because the linear model's high bias (underfitting) will dominate.

· Without knowing the exact shape of the true function or the noise variance, we cannot theoretically determine which model will have a lower test RSS.



## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 